# Misinformation Counter-Strategy & Threat Detector

This notebook implements the strategy to proactively identify, analyze, and neutralize misinformation campaigns targeting your reputation. It leverages the X API for monitoring and OpenAI for semantic analysis.

In [ ]:
# @title 1. Setup & Installation
!pip install tweepy openai pandas textblob

In [ ]:
# @title 2. Configuration & API Keys
import os
from getpass import getpass
import pandas as pd
import json
import random
from datetime import datetime

# API Keys (Leave blank to use Mock Data)
print("Enter X (Twitter) Bearer Token (Leave blank for Mock Mode):")
X_BEARER_TOKEN = getpass()

print("Enter OpenAI API Key (Leave blank for Mock Mode):")
OPENAI_API_KEY = getpass()

USE_MOCK_DATA = not (X_BEARER_TOKEN and OPENAI_API_KEY)

if USE_MOCK_DATA:
    print("\n⚠️ Missing API Keys. Switching to MOCK MODE.")
else:
    print("\n✅ API Keys detected. Switching to LIVE MODE.")
    import tweepy
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

## Phase 0: Calibration (Historical Data)
Before running on live data, we calibrate the system using historical Community Notes. We compare our AI's generated notes against real "Helpful" notes.

In [ ]:
# @title Download & Load Historical Data
# In a real scenario, we would download the latest dump from https://twitter.com/i/communitynotes/download-data
# For this notebook, we will create a small calibration dataset of real examples.

calibration_data = [
    {
        "tweet_text": "The moon landing was faked! Look at the shadows.",
        "real_note": "The moon landing is a well-documented historical event. The shadows are consistent with the light source on the moon. Source: NASA.",
        "verdict": "Misinformation"
    },
    {
        "tweet_text": "Drinking water causes dehydration if you don't add salt.",
        "real_note": "Drinking plain water hydrates the body effectively. While electrolytes are important, plain water does not cause dehydration. Source: Mayo Clinic.",
        "verdict": "Misinformation"
    }
]

print(f"Loaded {len(calibration_data)} calibration examples.")

## Phase 1: Surveillance (Data Ingestion)
We listen for mentions and keywords. If in Mock Mode, we generate synthetic tweets.

In [ ]:
def fetch_tweets(query, max_results=5):
    if USE_MOCK_DATA:
        # Generate Mock Tweets
        mock_tweets = [
            {"text": "I heard User X is a total scammer! They never pay their devs.", "author_id": "123", "id": "101", "public_metrics": {"like_count": 150, "retweet_count": 20}, "created_at": "2023-10-27T10:00:00Z"},
            {"text": "User X's new product is actually amazing. Love it!", "author_id": "124", "id": "102", "public_metrics": {"like_count": 5, "retweet_count": 0}, "created_at": "2023-10-27T10:05:00Z"},
            {"text": "Why is everyone saying User X stole the code? Is it true?", "author_id": "125", "id": "103", "public_metrics": {"like_count": 45, "retweet_count": 5}, "created_at": "2023-10-27T10:10:00Z"},
            {"text": "Just had a coffee. Nice day.", "author_id": "126", "id": "104", "public_metrics": {"like_count": 1, "retweet_count": 0}, "created_at": "2023-10-27T10:15:00Z"}
        ]
        return mock_tweets
    else:
        # Real API Call
        try:
            client_x = tweepy.Client(bearer_token=X_BEARER_TOKEN)
            tweets = client_x.search_recent_tweets(query=query, max_results=max_results, tweet_fields=['public_metrics', 'created_at', 'author_id'])
            if not tweets.data:
                return []
            return [t.data for t in tweets.data]
        except Exception as e:
            print(f"Error fetching tweets: {e}")
            return []

# Test the function
tweets = fetch_tweets("User X")
print(f"Fetched {len(tweets)} tweets.")

## Phase 2: Intelligence (Threat Analysis)
We use an LLM to analyze sentiment, extract claims, and score the threat level.

In [ ]:
# Truth Database (Simple Text for Context)
TRUTH_DB = """
FACTS ABOUT USER X:
1. User X pays all developers on time. All invoices from 2023-2024 are settled.
2. User X did not steal code; the codebase is open source under MIT license.
3. User X has never been banned from any platform.
"""

def analyze_threat(tweet_text, metrics=None):
    if metrics is None:
        metrics = {'like_count': 0, 'retweet_count': 0}
        
    if USE_MOCK_DATA:
        # Simple Mock Logic
        is_negative = "scammer" in tweet_text.lower() or "stole" in tweet_text.lower() or "faked" in tweet_text.lower()
        sentiment = "Negative" if is_negative else "Neutral"
        claim = "Financial Misconduct" if "scammer" in tweet_text else "IP Theft" if "stole" in tweet_text else "Conspiracy" if "faked" in tweet_text else "None"
        threat_score = 85 if is_negative else 10
        reasoning = "Contains trigger words" if is_negative else "Benign text"
    else:
        # OpenAI Analysis
        prompt = f"""
        Analyze this tweet for reputational threat against User X.
        Tweet: "{tweet_text}"
        
        Known Facts:
        {TRUTH_DB}
        
        Return JSON with keys: sentiment (Positive/Neutral/Negative), claim_extracted, fact_check_result, threat_score (0-100), reasoning.
        """
        try:
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[{"role": "system", "content": "You are a reputation management AI."}, {"role": "user", "content": prompt}],
                response_format={ "type": "json_object" }
            )
            result = json.loads(response.choices[0].message.content)
            sentiment = result.get('sentiment')
            claim = result.get('claim_extracted')
            threat_score = result.get('threat_score')
            reasoning = result.get('reasoning')
        except Exception as e:
            print(f"AI Error: {e}")
            return None

    # Adjust score based on virality
    virality_bonus = min(metrics['like_count'] / 10, 20) # Cap bonus at 20
    final_score = min(threat_score + virality_bonus, 100)
    
    return {
        "text": tweet_text,
        "sentiment": sentiment,
        "claim": claim,
        "threat_score": final_score,
        "reasoning": reasoning,
        "metrics": metrics
    }

# Test Analysis
analysis = analyze_threat(tweets[0]['text'], tweets[0]['public_metrics'])
print(json.dumps(analysis, indent=2))

## Phase 3: Counter-Measures (Community Notes)
If a threat is high, we draft a Community Note.

In [ ]:
def draft_community_note(analysis):
    if analysis['threat_score'] < 50:
        return "No action needed."
        
    if USE_MOCK_DATA:
        return f"DRAFT NOTE: The claim '{analysis['claim']}' is disputed. Public records show all obligations met. Source: [Link]"
    else:
        prompt = f"""
        Draft a Twitter Community Note to refute this claim based on the facts.
        Claim: {analysis['claim']}
        Facts: {TRUTH_DB}
        Style: Neutral, objective, helpful. Max 280 chars.
        """
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content

In [ ]:
# @title Run Calibration Loop
print("🔬 RUNNING CALIBRATION...\n")

for item in calibration_data:
    print(f"Tweet: {item['tweet_text']}")
    analysis = analyze_threat(item['tweet_text'])
    draft = draft_community_note(analysis)
    print(f"AI Draft: {draft}")
    print(f"Real Note: {item['real_note']}")
    print("-" * 40)

## War Room Dashboard
Run the full loop and display the results.

In [ ]:
# @title Run Threat Detector
results = []
print("🔍 Scanning for threats...\n")

for tweet in tweets:
    analysis = analyze_threat(tweet['text'], tweet['public_metrics'])
    if analysis:
        note = draft_community_note(analysis)
        analysis['draft_note'] = note
        results.append(analysis)

# Display as DataFrame
df = pd.DataFrame(results)
display_cols = ['threat_score', 'sentiment', 'claim', 'text', 'draft_note']
df_display = df[display_cols].sort_values(by='threat_score', ascending=False)

# Styling
def color_threat(val):
    color = 'red' if val > 75 else 'orange' if val > 50 else 'green'
    return f'color: {color}; font-weight: bold'

df_display.style.applymap(color_threat, subset=['threat_score'])